In [1]:
import pandas as pd
import numpy as np



In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
credits = pd.read_csv('/content/tmdb_5000_credits.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/tmdb_5000_credits.csv'

In [ ]:
movies = pd.read_csv('/content/tmdb_5000_movies.csv')

In [ ]:
movies.head()

In [ ]:
credits.head(1)['cast'].values

In [ ]:
movies = pd.read_csv('/content/tmdb_5000_movies.csv')
movies = movies.merge(credits, on='title')

WE Have 2 data frames , so we are going  to merge both data frames

In [ ]:
credits.shape

In [ ]:
movies.head(1)

WE are going to remove the columns that are useless in recommendation . If it will help in creating tags .

In [ ]:
#genre
#id
#keywords
#title
#overview
#cast
#crew
#try release date sometime (numerical column)
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [ ]:
movies.info()

In [ ]:
movies.head()

In [ ]:
movies.isnull().sum()

In [ ]:
movies.dropna(inplace=True)

In [ ]:
movies.duplicated().sum()

In [ ]:
movies.iloc[0].genres

In [ ]:
#[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]
#['Action',"Adventure",'Fantasy','Science fiction']
#import ast
#ast.literal_eval  ==== used to convert string into list

In [ ]:
import ast
def convert(obj):
  L=[]
  for i in ast.literal_eval(obj):
    L.append(i['name'])
  return L
  #helper function

In [ ]:
movies['genres'] = movies['genres'].apply(convert)

In [ ]:
movies.head()

In [ ]:
movies['keywords']=movies['keywords'].apply(convert)

In [ ]:
movies.head()

In [ ]:
movies['cast'][0]

In [ ]:

def convert3(obj):
  L=[]
  counter = 0
  for i in ast.literal_eval(obj):
    if counter !=3:
      L.append(i['name'])
      counter+=1
    else:
       break
  return L
  #helper function

In [ ]:
movies['cast'] = movies['cast'].apply(convert3)

In [ ]:
movies.head()

In [ ]:
movies['crew'][0]

In [ ]:

def fetch_director(obj):
  L=[]
  for i in ast.literal_eval(obj):
    if i['job']=='Director':
      L.append(i['name'])
      break
  return L

In [ ]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies.head()

In [ ]:
movies['overview'][0]

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [ ]:
movies.head()

In [ ]:
#need to remove spaces  Sam Worthington == SamWorthington
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [ ]:
movies.head()

In [ ]:
movies['tags']=movies['overview']+ movies['genres']+ movies['keywords']+movies['cast']+movies['crew']


In [ ]:
movies.head()

In [ ]:
new_df = movies[['movie_id', 'title','tags']]

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

In [ ]:
new_df.head()

In [ ]:
new_df['tags'][0]

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

In [ ]:
new_df.head()

In [ ]:
#vectorization of strings
#extract most repeated words,every movie is a vector, start with less words
#do not consider stopwords = used in formation of sentences in eng, does not contribute in the meaning of sentence

In [ ]:
from sklearn.feature_extraction .text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')
#Vector that counVector returns is a sparse matrix , we will change it to numpy array cuz that is needed

In [ ]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [ ]:
vectors[0]

In [ ]:
len(cv.get_feature_names_out())
#common words

In [ ]:
cv.get_feature_names_out()

In [ ]:
#apply stemming = converts words into their root words
#loved,loving, love
#love,love,love


In [ ]:
import nltk



In [ ]:
!pip install nltk

In [ ]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [ ]:
def stem(text):
  y=[]
  for i in text.split():
    y.append(ps.stem(i))

  return " ".join(y)

In [ ]:
stem('In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy ScienceFiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d SamWorthington ZoeSaldana SigourneyWeaver JamesCameron')

In [ ]:
new_df['tags'] = new_df['tags'].apply(stem)

In [ ]:
#calculate distance of every movie with every another movie
#greater distance == less similar movies (distance is inversely proportional to similarity)
#not euclidian distance(tip to tip distance) but cosine distance( angular distance)
#jitna high dimesional data data me kaam karte ho == euclidian distance fail karta  h (curse of dimensionality)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
similarity = cosine_similarity(vectors)

In [ ]:
sorted(list(enumerate(similarity[0])),reverse = True, key=lambda x:x[1])[1:6]
#diagonal of this matrix will alwaYS BE 1
#HAR MOVIES ka usi ke saath similarity

In [ ]:
def recommend(movies):
  movie_index = new_df[new_df['title'] == movies].index[0]
  distances = similarity[movie_index]
  movies_list = sorted(list(enumerate(distances)),reverse = True, key=lambda x:x[1])[1:6]
  for i in movies_list:
    print(new_df.iloc[i[0]].title)


In [ ]:
#sorted (similarity[0],reverse=True)
#jaise hee sorting kari to index loose hogya fir enumerate function use kiya


In [ ]:
recommend('The Big Bounce')

In [ ]:
new_df.iloc[999].title

In [ ]:
import pickle

In [ ]:
pickle.dump(new_df,open('/content/tmdb_5000_movies.csv.pkl','wb'))

In [ ]:
new_df['title'].values

In [ ]:
pickle.dump(similarity,open('/content/similarity.pkl','wb'))